# Preprocessing and Data Leakage

Build a deliberately messy dataset, handle it with a ColumnTransformer,
and measure how much data leakage inflates scores.

Main question: how wrong is your score if you preprocess before splitting?

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error

RANDOM_STATE = 42

rng = np.random.default_rng(RANDOM_STATE)

In [2]:
n = 2000

df = pd.DataFrame({
    "income": rng.normal(50000, 15000, n),
    "age": rng.integers(18, 80, n),
    "score": rng.normal(0, 1, n),
    "city": rng.choice(["Delhi", "Mumbai", "Pune", "Kochi"], n),
    "plan": rng.choice(["basic", "silver", "gold"], n),
})

# Known relationship
df["target"] = (
    0.00004 * df["income"]
    + 0.03 * df["age"]
    + 0.8 * df["score"]
    + rng.normal(0, 0.5, n)
)

In [3]:
# Missing values
df.loc[
    rng.choice(n, 200, replace=False),
    "income"
] = np.nan

df.loc[
    rng.choice(n, 150, replace=False),
    "city"
] = np.nan

# Outliers
df.loc[
    rng.choice(n, 20, replace=False),
    "income"
] *= 50

# Feature on a wildly different scale
df["account_balance"] = rng.normal(2_000_000, 800_000, n)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   income           1800 non-null   float64
 1   age              2000 non-null   int64  
 2   score            2000 non-null   float64
 3   city             1850 non-null   str    
 4   plan             2000 non-null   str    
 5   target           2000 non-null   float64
 6   account_balance  2000 non-null   float64
dtypes: float64(4), int64(1), str(2)
memory usage: 109.5 KB


In [5]:
df.describe()

,income,age,score,target,account_balance
count,1.800000e+03,2000.000000,2000.000000,2000.000000,2.000000e+03
mean,7.413746e+04,48.198500,-0.007481,3.405431,1.983821e+06
std,2.453358e+05,17.776589,0.991149,1.234045,8.103278e+05
min,-4.726192e+03,18.000000,-3.143666,-1.147968,-4.955799e+05
25%,3.950052e+04,33.000000,-0.687316,2.547575,1.454478e+06
50%,4.987476e+04,48.000000,-0.019370,3.374412,1.965410e+06
75%,5.927629e+04,64.000000,0.626710,4.257615,2.529968e+06
max,3.652724e+06,79.000000,3.454046,7.297122,4.559154e+06


## Data problems

The dataset contains several deliberate problems:

- Missing values in `income` and `city`.
- Extreme outliers in `income`.
- Features with very different numerical scales.
- Categorical features (`city` and `plan`) that need encoding.

Missing values affect models because most estimators cannot work directly
with NaNs. Outliers can strongly affect scale-sensitive models such as
Linear Regression and Ridge. Different feature scales matter particularly
for models using distances or coefficient regularization. Categorical
features must be converted into numerical representations before most
scikit-learn models can use them.

In [6]:
X = df.drop(columns="target")
y = df["target"]

In [7]:
X_bad = X.copy()

# Fill missing values using the entire dataset
X_bad["income"] = X_bad["income"].fillna(
    X_bad["income"].median()
)

X_bad["city"] = X_bad["city"].fillna(
    X_bad["city"].mode()[0]
)

# One-hot encode categorical columns
X_bad = pd.get_dummies(
    X_bad,
    columns=["city", "plan"]
)

# Scale using the entire dataset
scaler = StandardScaler()

X_bad[X_bad.columns] = scaler.fit_transform(X_bad)

In [8]:
Xtr_b, Xte_b, ytr_b, yte_b = train_test_split(
    X_bad,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

In [9]:
m = Ridge()

m.fit(Xtr_b, ytr_b)

leaky_preds = m.predict(Xte_b)

leaky_rmse = root_mean_squared_error(
    yte_b,
    leaky_preds
)

print(f"Leaky RMSE: {leaky_rmse:.4f}")

leakage_results = [
    {
        "version": "A - Leaky preprocessing",
        "rmse": leaky_rmse
    }
]

Leaky RMSE: 0.7940


In [10]:
num_cols = [
    "income",
    "age",
    "score",
    "account_balance"
]

cat_cols = [
    "city",
    "plan"
]

In [11]:
preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]),
        num_cols
    ),
    (
        "cat",
        Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore"))
        ]),
        cat_cols
    )
])

In [12]:
pipe = Pipeline([
    ("prep", preprocessor),
    ("model", Ridge())
])

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

In [14]:
pipe.fit(X_train, y_train)

clean_preds = pipe.predict(X_test)

clean_rmse = root_mean_squared_error(
    y_test,
    clean_preds
)

print(f"Clean Pipeline RMSE: {clean_rmse:.4f}")

leakage_results.append({
    "version": "B - Clean Pipeline",
    "rmse": clean_rmse
})

Clean Pipeline RMSE: 0.7940


In [15]:
leaky_cv_scores = cross_val_score(
    Ridge(),
    X_bad,
    y,
    scoring="neg_root_mean_squared_error",
    cv=5
)

leaky_cv_rmse = -leaky_cv_scores

print("Leaky CV RMSE scores:")
print(leaky_cv_rmse)

print(
    f"Mean Leaky CV RMSE: "
    f"{leaky_cv_rmse.mean():.4f}"
)

Leaky CV RMSE scores:
[0.73138403 0.79236074 0.77759014 0.79554083 0.78136607]
Mean Leaky CV RMSE: 0.7756


In [16]:
clean_cv_scores = cross_val_score(
    pipe,
    X,
    y,
    scoring="neg_root_mean_squared_error",
    cv=5
)

clean_cv_rmse = -clean_cv_scores

print("Clean CV RMSE scores:")
print(clean_cv_rmse)

print(
    f"Mean Clean CV RMSE: "
    f"{clean_cv_rmse.mean():.4f}"
)

leakage_results.append({
    "version": "C - Leaky 5-fold CV",
    "rmse": leaky_cv_rmse.mean()
})

leakage_results.append({
    "version": "D - Clean 5-fold CV",
    "rmse": clean_cv_rmse.mean()
})

Clean CV RMSE scores:
[0.73137559 0.79235991 0.77758761 0.79553929 0.78136382]
Mean Clean CV RMSE: 0.7756


In [17]:
leakage_df = pd.DataFrame(leakage_results)

leakage_df

leakage_df.sort_values("rmse")

,version,rmse
3,D - Clean 5-fold CV,0.775645
2,C - Leaky 5-fold CV,0.775648
1,B - Clean Pipeline,0.794003
0,A - Leaky preprocessing,0.794013


In [18]:
holdout_gap = leaky_rmse - clean_rmse
cv_gap = leaky_cv_rmse.mean() - clean_cv_rmse.mean()

print(f"Holdout RMSE difference: {holdout_gap:.6f}")
print(f"CV RMSE difference:      {cv_gap:.6f}")

Holdout RMSE difference: 0.000010
CV RMSE difference:      0.000003


## Leakage analysis

Version A preprocesses the entire dataset before splitting. This allows
information from the eventual test set to influence preprocessing
parameters such as the median, mode, and scaling statistics.

Version B splits the raw data first and puts preprocessing inside a Pipeline.
The preprocessing steps therefore learn their parameters only from the
training data.

The difference in RMSE is the measured effect of this leakage on this
dataset. Even if the difference is small, the workflow in Version A is
incorrect because the evaluation data has influenced the training process.

The Pipeline approach is safer because the preprocessing steps are
structurally connected to model fitting and are automatically fitted only
on the training portion during evaluation and cross-validation.

Leakage can become much more damaging when preprocessing uses information
strongly related to the target or future information, such as target
encoding, time-series features, or statistics calculated using future
observations.

In [19]:
ordinal_preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]),
        num_cols
    ),
    (
        "cat",
        Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ))
        ]),
        cat_cols
    )
])

ordinal_pipe = Pipeline([
    ("prep", ordinal_preprocessor),
    ("model", Ridge())
])

In [20]:
ordinal_pipe.fit(X_train, y_train)

ordinal_preds = ordinal_pipe.predict(X_test)

ordinal_rmse = root_mean_squared_error(
    y_test,
    ordinal_preds
)

print(f"OrdinalEncoder RMSE: {ordinal_rmse:.4f}")

OrdinalEncoder RMSE: 0.7924


In [21]:
onehot_preds = pipe.predict(X_test)

onehot_rmse = root_mean_squared_error(
    y_test,
    onehot_preds
)

print(f"OneHotEncoder RMSE: {onehot_rmse:.4f}")

print(f"OrdinalEncoder RMSE: {ordinal_rmse:.4f}")
print(f"OneHotEncoder RMSE:  {onehot_rmse:.4f}")

OneHotEncoder RMSE: 0.7940
OrdinalEncoder RMSE: 0.7924
OneHotEncoder RMSE:  0.7940


In [22]:
X_weird = X.iloc[:5].copy()

X_weird["city"] = "Chennai"

X_weird

,income,age,score,city,plan,account_balance
0,54570.756196,63,-1.344569,Chennai,silver,5.778640e+05
1,34400.238406,20,-0.784259,Chennai,basic,3.129755e+06
2,61256.767937,40,-0.125044,Chennai,basic,2.127773e+06
3,64108.470746,45,-0.610099,Chennai,silver,1.964926e+06
4,20734.472170,77,2.299664,Chennai,basic,1.172187e+06


In [23]:
print(pipe.predict(X_weird))

[2.6994311  1.92767419 3.0712401  2.76616375 6.16804886]


In [24]:
unsafe_preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]),
        num_cols
    ),
    (
        "cat",
        Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="error"))
        ]),
        cat_cols
    )
])

strict_pipe = Pipeline([
    ("prep", unsafe_preprocessor),
    ("model", Ridge())
])

strict_pipe.fit(X_train, y_train)

strict_pipe.predict(X_weird)

ValueError: Found unknown categories ['Chennai'] in column 0 during transform

## Encoder choice

OrdinalEncoder scored 0.7924 versus OneHotEncoder's 0.7940. This
difference is not evidence that ordinal encoding is better. `city` and
`plan` are not part of the target formula in this synthetic data, so both
encodings represent pure noise. One-hot adds 7 noise columns while ordinal
adds 2, so the ordinal model simply has slightly less noise to overfit.

The experiment cannot answer the encoder question because the dataset was
not built to test it. To do that properly, `city` would need a real effect
on the target.

In [25]:
leakage_results.append({
    "version": "E - OrdinalEncoder",
    "rmse": ordinal_rmse
})

leakage_results.append({
    "version": "F - OneHotEncoder",
    "rmse": onehot_rmse
})

final_comparison = pd.DataFrame(leakage_results)

final_comparison

,version,rmse
0,A - Leaky preprocessing,0.794013
1,B - Clean Pipeline,0.794003
2,C - Leaky 5-fold CV,0.775648
3,D - Clean 5-fold CV,0.775645
4,E - OrdinalEncoder,0.792425
5,F - OneHotEncoder,0.794003


In [26]:
final_comparison.to_csv(
    "../reports/leakage_comparison.csv",
    index=False
)

In [27]:
from sklearn.feature_selection import SelectKBest, f_regression

# Create 200 completely random noise features
noise = pd.DataFrame(
    rng.normal(0, 1, (n, 200)),
    columns=[f"noise_{i}" for i in range(200)]
)

X_noise = pd.concat([
    df[["income", "age", "score"]].fillna(0),
    noise
], axis=1)

# WRONG: select features using the entire dataset before splitting
selector = SelectKBest(f_regression, k=10).fit(X_noise, y)
X_sel = selector.transform(X_noise)

Xtr, Xte, ytr, yte = train_test_split(
    X_sel,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

leaky_fs = root_mean_squared_error(
    yte,
    Ridge().fit(Xtr, ytr).predict(Xte)
)

# RIGHT: feature selection happens inside the pipeline
Xtr2, Xte2, ytr2, yte2 = train_test_split(
    X_noise,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

clean_pipe_fs = Pipeline([
    ("select", SelectKBest(f_regression, k=10)),
    ("model", Ridge())
]).fit(Xtr2, ytr2)

clean_fs = root_mean_squared_error(
    yte2,
    clean_pipe_fs.predict(Xte2)
)

print(f"Leaky feature selection RMSE: {leaky_fs:.4f}")
print(f"Clean feature selection RMSE: {clean_fs:.4f}")

leakage_results.append({
    "version": "G - Leaky feature selection",
    "rmse": leaky_fs
})

leakage_results.append({
    "version": "H - Clean feature selection",
    "rmse": clean_fs
})

Leaky feature selection RMSE: 0.7828
Clean feature selection RMSE: 0.7924


In [28]:
leakage_df = pd.DataFrame(leakage_results)

leakage_df.sort_values("rmse")

,version,rmse
3,D - Clean 5-fold CV,0.775645
2,C - Leaky 5-fold CV,0.775648
6,G - Leaky feature selection,0.782764
7,H - Clean feature selection,0.792401
4,E - OrdinalEncoder,0.792425
1,B - Clean Pipeline,0.794003
5,F - OneHotEncoder,0.794003
0,A - Leaky preprocessing,0.794013


In [29]:
try:
    strict_pipe.predict(X_weird)
except ValueError as e:
    print("Failed as expected:", e)

Failed as expected: Found unknown categories ['Chennai'] in column 0 during transform


In [30]:
pd.DataFrame(leakage_results).to_csv(
    "../reports/leakage_comparison.csv",
    index=False
)

## Conclusion

### Leakage from imputation and scaling was negligible

The leaky approach scored an RMSE of 0.794013 against 0.794003 for the
clean Pipeline — a difference of 0.000010. The 5-fold cross-validation
comparison showed the same thing: 0.775648 leaky against 0.775645 clean.

The reason is that medians, modes and scaling statistics barely change
whether they are computed on 1,600 rows or 2,000. The leak was real but
carried almost no information. The workflow is still wrong, but on this
data it was not costly.

### Leakage from feature selection was much larger

Leaky feature selection scored 0.7828 against 0.7924 for the clean
version — a gap of 0.0096, nearly a thousand times larger than the
imputation leak.

The cause is that SelectKBest ranks features by their correlation with
the target. Running it on the full dataset means the test rows helped
decide which of the 200 noise columns looked useful.

Note the direction: the leaky version scored *better*. Leakage does not
announce itself as a bug — it announces itself as an improvement. A model
selected this way would promise 0.7828 and deliver 0.7924 in production.
That is what makes it easy to ship by accident.

### The rule this suggests

Leakage is dangerous in proportion to how much the leaked step depends on
the target. Unsupervised steps — scaling, imputation, encoding — leak
little. Any step that looks at `y` — feature selection, target encoding,
resampling — leaks a lot.

### Unseen categories

With `handle_unknown="ignore"`, the pipeline predicted successfully for an
unseen city. With `handle_unknown="error"`, it raised a ValueError.

Neither is automatically correct. Ignoring is right when new categories are
expected and a prediction is still wanted. Failing loudly is right when an
unseen category signals a schema or data-quality problem, because a silent
prediction built on a dropped feature is worse than no prediction at all.

### Why pipelines matter

A Pipeline is not just convenience. It makes correct behaviour structural
rather than a matter of discipline: each preprocessing step is fitted only
on the training portion, automatically, including inside every
cross-validation fold. The alternative is remembering to do it right every
single time.